# Job Salary Estimator — DAY 2: Data Pre-processing

Use LLMs to rewrite job descriptions into a concise, structured format.

In [ ]:
from litellm import completion
from dotenv import load_dotenv
import json
from job_salary.batch import Batch
from job_salary.items import Job

load_dotenv(override=True)

Set `LITE_MODE = True` for a small subset (fast, cheap). Set `False` for full dataset.

In [ ]:
LITE_MODE = True

In [ ]:
# Load curated data - use pre-pushed hub dataset or local train/val/test from Day 1
# Option: Job.from_hub('username/job_salaries')
# For now, run Day 1 first and use train, val, test from there

from datasets import load_dataset
import os

# If you pushed to hub:
# train, val, test = Job.from_hub('your_username/job_salaries')
# items = train + val + test

# Or load a sample for demo:
try:
    ds = load_dataset('lukebarousse/data_jobs', split='train', trust_remote_code=True)
    from job_salary.parser import parse
    from tqdm.notebook import tqdm
    items = [parse(dp) for dp in tqdm(ds.select(range(5000)))]
    items = [j for j in items if j is not None]
    print(f'Loaded {len(items):,} jobs')
except Exception as e:
    print(f'Load data first (Day 1 or hub): {e}')
    items = []

In [ ]:
for i, j in enumerate(items):
    j.id = i

In [ ]:
SYSTEM_PROMPT = """Create a concise job description. Respond only in this format.
Title: Short job title
Category: Role type (e.g. Data Scientist, Engineer)
Company: Company name
Location: Location if known
Remote: Yes/No
Description: 1-2 sentence summary of the role and key requirements"""

print(items[0].full)

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full}
]
response = completion(messages=messages, model="groq/llama-3.3-70b-versatile")
print(response.choices[0].message.content)

In [ ]:
# Batch processing via Groq (optional - requires GROQ_API_KEY)
# Batch.create(items[:1000], LITE_MODE)  # 1000 for demo
# Batch.run()
# Batch.fetch()

# Or run inline for small set:
from tqdm.notebook import tqdm
subset = items[:200]  # small demo
for j in tqdm(subset):
    try:
        r = completion(messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": j.full}], model="groq/llama-3.3-70b-versatile")
        j.summary = r.choices[0].message.content
    except Exception as e:
        j.summary = j.full[:500]  # fallback

print(subset[0].summary)

In [ ]:
# Push processed dataset (with summaries) to hub
# for j in items: j.full = None; j.id = None
# train, val, test = items[:int(0.8*len(items))], items[int(0.8*len(items)):int(0.9*len(items))], items[int(0.9*len(items)):]
# Job.push_to_hub('username/job_salaries_processed', train, val, test)